<a href="https://colab.research.google.com/github/Divyansh-yecho/Lunar_lander_toy_model/blob/main/lunar_lander.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Lunar Lander — DQN with Stable-Baselines3

Training a Deep Q-Network (DQN) agent to solve the `LunarLander-v3` environment from Gymnasium.

**Highlights:**
- DQN with experience replay and target network
- Checkpoint saving every 10k steps
- Training logs written to file (not cluttering the notebook)
- Video recording of the final agent

## 1. Install Dependencies

In [1]:
!pip install -q gymnasium[box2d] stable-baselines3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 17.2 MB/s eta 0:00:00


## 2. Imports

In [2]:
import os
import logging

import gymnasium as gym
import numpy as np

from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from gymnasium.wrappers import RecordVideo

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 3. Mount Google Drive & Set Up Directories

In [3]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR      = "/content/drive/MyDrive/lunar_lander_project"
CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints")
MODEL_DIR      = os.path.join(BASE_DIR, "models")
LOG_DIR        = os.path.join(BASE_DIR, "logs")
VIDEO_DIR      = os.path.join(BASE_DIR, "videos")

for d in [CHECKPOINT_DIR, MODEL_DIR, LOG_DIR, VIDEO_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅ Directories ready")

Mounted at /content/drive
✅ Directories ready


## 4. Configure File Logging

All verbose training output is redirected to `training.log` so the notebook stays clean.

In [4]:
log_file = os.path.join(LOG_DIR, "training.log")

logging.basicConfig(
    filename=log_file,
    filemode="a",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

# Silence stable-baselines3 stdout by routing its logger to the file handler
sb3_logger = logging.getLogger("stable_baselines3")
sb3_logger.setLevel(logging.INFO)
sb3_logger.propagate = True  # uses the root handler above

print(f"📝 Logs → {log_file}")

📝 Logs → /content/drive/MyDrive/lunar_lander_project/logs/training.log


## 5. Create Environment

In [5]:
env = gym.make("LunarLander-v3")
env = Monitor(env, filename=os.path.join(LOG_DIR, "monitor"))

print(f"Observation space : {env.observation_space}")
print(f"Action space      : {env.action_space}")

<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute
<frozen importlib._bootstrap>:488: DeprecationWarning: builtin type swigvarlink has no __module__ attribute
/usr/local/lib/python3.12/dist-packages/pygame/pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/usr/local/lib/python3.12/dist-packages/

Observation space : Box([ -2.5        -2.5       -10.        -10.         -6.2831855 -10.
  -0.         -0.       ], [ 2.5        2.5       10.        10.         6.2831855 10.
  1.         1.       ], (8,), float32)
Action space      : Discrete(4)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 6. Build DQN Model

In [6]:
model = DQN(
    "MlpPolicy",
    env,
    learning_rate=5e-4,
    buffer_size=100_000,
    learning_starts=5_000,
    batch_size=64,
    gamma=0.99,
    train_freq=4,
    target_update_interval=1_000,
    exploration_fraction=0.15,
    exploration_final_eps=0.02,
    verbose=0,                          # ← suppressed; output goes to log file
    tensorboard_log=LOG_DIR,
)

print("✅ Model ready")

✅ Model ready


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 7. Train — Phase 1 (200k steps)

In [7]:
checkpoint_cb = CheckpointCallback(
    save_freq=10_000,
    save_path=CHECKPOINT_DIR,
    name_prefix="dqn_lunar",
)

print("🚀 Training phase 1 …")
model.learn(total_timesteps=200_000, callback=checkpoint_cb, progress_bar=True)

mean_r, std_r = evaluate_policy(model, env, n_eval_episodes=10)
logging.info(f"Phase 1 eval — mean: {mean_r:.2f}, std: {std_r:.2f}")
print(f"Phase 1 → mean reward: {mean_r:.2f} ± {std_r:.2f}")

Output()

/usr/local/lib/python3.12/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

🚀 Training phase 1 …


Phase 1 → mean reward: 54.69 ± 113.17


## 8. Continue Training — Phase 2 (300k steps)

In [8]:
# Load best checkpoint and keep training
ckpt_path = os.path.join(CHECKPOINT_DIR, "dqn_lunar_200000_steps")
model = DQN.load(ckpt_path, env=env)

print("🚀 Training phase 2 …")
model.learn(total_timesteps=300_000, callback=checkpoint_cb, progress_bar=True)

mean_r, std_r = evaluate_policy(model, env, n_eval_episodes=10)
logging.info(f"Phase 2 eval — mean: {mean_r:.2f}, std: {std_r:.2f}")
print(f"Phase 2 → mean reward: {mean_r:.2f} ± {std_r:.2f}")

Output()

🚀 Training phase 2 …


Phase 2 → mean reward: 70.39 ± 110.24


## 9. Continue Training — Phase 3 (300k steps)

In [9]:
print("🚀 Training phase 3 …")
model.learn(total_timesteps=300_000, callback=checkpoint_cb, progress_bar=True)

mean_r, std_r = evaluate_policy(model, env, n_eval_episodes=20)
logging.info(f"Phase 3 final eval — mean: {mean_r:.2f}, std: {std_r:.2f}")
print(f"Final → mean reward: {mean_r:.2f} ± {std_r:.2f}")

model.save(os.path.join(MODEL_DIR, "final_model"))
print("💾 Model saved")

Output()

🚀 Training phase 3 …


Final → mean reward: 199.56 ± 104.13
💾 Model saved


## 10. Record Agent Videos

In [10]:
video_env = gym.make("LunarLander-v3", render_mode="rgb_array")
video_env = RecordVideo(
    video_env,
    video_folder=VIDEO_DIR,
    episode_trigger=lambda e: True,  # record every episode
    name_prefix="lunar_agent",
)

for episode in range(2):            # record 2 episodes for GitHub
    obs, _ = video_env.reset()
    while True:
        action, _ = model.predict(obs, deterministic=True)
        obs, _, done, truncated, _ = video_env.step(action)
        if done or truncated:
            break

video_env.close()
print(f"🎬 Videos saved to {VIDEO_DIR}")
print(os.listdir(VIDEO_DIR))

/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /content/drive/MyDrive/lunar_lander_project/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


🎬 Videos saved to /content/drive/MyDrive/lunar_lander_project/videos
['rl-video-episode-1.mp4', 'rl-video-episode-0.mp4', 'lunar_agent-episode-0.mp4', 'lunar_agent-episode-1.mp4']


## 11. Playback Videos in Notebook

In [11]:
from IPython.display import HTML, display
from base64 import b64encode

def show_video(path):
    mp4 = open(path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f'''
        <video width="400" controls>
            <source src="{data_url}" type="video/mp4">
        </video>
    '''))

video_files = sorted(
    [f for f in os.listdir(VIDEO_DIR) if f.endswith(".mp4")]
)

for vf in video_files[:2]:
    print(f"▶ {vf}")
    show_video(os.path.join(VIDEO_DIR, vf))

▶ lunar_agent-episode-0.mp4


▶ lunar_agent-episode-1.mp4


## 12. View Training Log (last 20 lines)

In [12]:
with open(log_file) as f:
    lines = f.readlines()

print(f"Total log lines: {len(lines)}")
print("\n--- Last 20 entries ---")
print("".join(lines[-20:]))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/lunar_lander_project/logs/training.log'